<a href="https://colab.research.google.com/github/AK888-hp/ML-II_Lab/blob/main/foil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import pandas as pd
import math

# ============================================================
# 1. DATASET
# ============================================================

data = [
    ['Cloudy', 'Warm', 'Normal', 'Low', 'Yes'],
    ['Cloudy', 'Cold', 'High', 'High', 'No'],
    ['Clear', 'Warm', 'High', 'Low', 'No'],
    ['Stormy', 'Cold', 'Normal', 'High', 'No'],
    ['Stormy', 'Warm', 'Normal', 'Low', 'Yes'],
    ['Clear', 'Cold', 'Normal', 'Low', 'Yes'],
    ['Cloudy', 'Warm', 'High', 'High', 'No'],
    ['Clear', 'Warm', 'Normal', 'Low', 'Yes'],
    ['Stormy', 'Cold', 'High', 'Low', 'No'],
    ['Cloudy', 'Cold', 'Normal', 'Low', 'Yes'],
    ['Clear', 'Cold', 'High', 'High', 'No'],
    ['Stormy', 'Warm', 'High', 'High', 'No'],
    ['Cloudy', 'Warm', 'Normal', 'High', 'Yes'],
    ['Clear', 'Warm', 'High', 'Low', 'No']
]

columns = ['Sky', 'Temperature', 'Moisture', 'Wind', 'Activity']

df = pd.DataFrame(data, columns=columns)

print("DATASET")
print(df)


# ============================================================
# 2. SIMPLE CN2
# ============================================================

def cn2(df):

    rules = []

    # Learn rules for Yes examples
    while True:

        positives = df[df['Activity'] == 'Yes']

        if len(positives) == 0:
            break

        best_rule = None
        best_accuracy = 0

        for feature in columns[:-1]:

            for value in df[feature].unique():

                subset = df[df[feature] == value]

                if len(subset) == 0:
                    continue

                accuracy = (
                    len(subset[subset['Activity'] == 'Yes'])
                    / len(subset)
                )

                if accuracy > best_accuracy:
                    best_accuracy = accuracy
                    best_rule = (feature, value)

        if best_rule is None:
            break

        feature, value = best_rule

        rule = f"IF {feature} = {value} THEN Activity = Yes"

        rules.append(rule)

        # Remove covered examples
        df = df[df[feature] != value]

    return rules


# ============================================================
# 3. RUN CN2
# ============================================================

cn2_rules = cn2(df.copy())

print("\nCN2 RULES")
print("=" * 50)

for i, rule in enumerate(cn2_rules, 1):
    print(f"Rule {i}: {rule}")


# ============================================================
# 4. SIMPLE FOIL
# ============================================================

def foil(df):

    features = columns[:-1]

    positives = df[df['Activity'] == 'Yes']
    negatives = df[df['Activity'] == 'No']

    rules = []

    # Try every feature-value pair
    for feature in features:

        for value in df[feature].unique():

            subset = df[df[feature] == value]

            pos = len(subset[subset['Activity'] == 'Yes'])
            neg = len(subset[subset['Activity'] == 'No'])

            # A pure positive condition is a good FOIL rule
            if pos > 0 and neg == 0:

                rule = (
                    f"Activity(X) :- "
                    f"{feature}(X, {value})"
                )

                rules.append(rule)

    return rules


# ============================================================
# 5. RUN FOIL
# ============================================================

foil_rules = foil(df)

print("\nFOIL FIRST-ORDER RULES")
print("=" * 50)

for i, rule in enumerate(foil_rules, 1):
    print(f"Rule {i}: {rule}")


# ============================================================
# 6. PREDICTION
# ============================================================

def predict(sky, temperature, moisture, wind):

    # Rules derived from the dataset

    if sky == "Cloudy" and moisture == "Normal":
        return "Yes"

    if sky == "Clear" and temperature == "Cold":
        return "Yes"

    if sky == "Stormy" and temperature == "Warm":
        return "Yes"

    return "No"


# ============================================================
# 7. TEST PREDICTIONS
# ============================================================

print("\nPREDICTIONS")
print("=" * 50)

test_data = [
    ['Cloudy', 'Warm', 'Normal', 'Low'],
    ['Clear', 'Cold', 'Normal', 'High'],
    ['Stormy', 'Cold', 'High', 'Low'],
    ['Stormy', 'Warm', 'Normal', 'Low']
]

for example in test_data:

    result = predict(*example)

    print(
        f"{example} -> Activity = {result}"
    )


DATASET
       Sky Temperature Moisture  Wind Activity
0   Cloudy        Warm   Normal   Low      Yes
1   Cloudy        Cold     High  High       No
2    Clear        Warm     High   Low       No
3   Stormy        Cold   Normal  High       No
4   Stormy        Warm   Normal   Low      Yes
5    Clear        Cold   Normal   Low      Yes
6   Cloudy        Warm     High  High       No
7    Clear        Warm   Normal   Low      Yes
8   Stormy        Cold     High   Low       No
9   Cloudy        Cold   Normal   Low      Yes
10   Clear        Cold     High  High       No
11  Stormy        Warm     High  High       No
12  Cloudy        Warm   Normal  High      Yes
13   Clear        Warm     High   Low       No

CN2 RULES
Rule 1: IF Moisture = Normal THEN Activity = Yes

FOIL FIRST-ORDER RULES

PREDICTIONS
['Cloudy', 'Warm', 'Normal', 'Low'] -> Activity = Yes
['Clear', 'Cold', 'Normal', 'High'] -> Activity = Yes
['Stormy', 'Cold', 'High', 'Low'] -> Activity = No
['Stormy', 'Warm', 'Normal', 'L